# Train the guitar chord classifier on Colab

Enable a GPU runtime. Copy your current project folder to Google Drive under `My Drive`, including `pyproject.toml`, `src/`, and `data/raw/`. The raw dataset should contain `data.yaml` and the original `train/`, `valid/`, and `test/` directories, each with `images/` and `labels/`.

Open this notebook in Colab and run the cells in order. Update `PROJECT_ROOT` if you put the project elsewhere in Drive.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Guitar-Chord-Detection-Computer-Vision-Model')
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
ARTIFACTS = PROJECT_ROOT / 'artifacts'

required = [PROJECT_ROOT / 'pyproject.toml', PROJECT_ROOT / 'src' / 'guitar_chords' / 'cli.py',
            DATA_ROOT / 'data.yaml']
required += [DATA_ROOT / split / folder
             for split in ('train', 'valid', 'test') for folder in ('images', 'labels')]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing project or dataset paths:\n' + '\n'.join(missing))

## Install and check the runtime

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', str(PROJECT_ROOT)], check=True)

gpu_check = '''
import torch

if not torch.cuda.is_available():
    raise SystemExit('Enable a GPU runtime in Colab, then rerun this cell.')
print('Training device:', torch.cuda.get_device_name(0))
'''
subprocess.run([sys.executable, '-c', gpu_check], check=True)

## Train and save the best checkpoint

The training split updates the classifier. Validation macro F1 selects the best checkpoint, which is saved to Drive in a new run folder. The test split is reserved for the optional evaluation cell.

In [ ]:
from datetime import datetime, timezone

EPOCHS = 10
BATCH_SIZE = 32
SEED = 42

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
checkpoint = ARTIFACTS / run_id / 'model.pt'
print(f'Checkpoint: {checkpoint}', flush=True)

command = [
    sys.executable, '-u', '-m', 'guitar_chords.cli', 'train',
    '--data', str(DATA_ROOT), '--checkpoint', str(checkpoint),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--seed', str(SEED),
]
with subprocess.Popen(command, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'Training failed (exit {exit_code}). See the command output above.')

## Evaluate the held-out test set

Once the model and training settings are final, set `RUN_TEST_EVALUATION` to `True` and run this cell. To evaluate an earlier run, set `checkpoint` to its saved Drive path. Keep the dataset at the same Drive path used during training.

In [ ]:
RUN_TEST_EVALUATION = False

if RUN_TEST_EVALUATION:
    subprocess.run([
        sys.executable, '-m', 'guitar_chords.cli', 'evaluate',
        '--checkpoint', str(checkpoint), '--batch-size', str(BATCH_SIZE),
    ], check=True)